In [1]:
import requests
import pandas as pd
import hashlib
from pathlib import Path

In [4]:
df = pd.read_csv("/Users/mustafazayyat/Downloads/tcp.csv")

/var/folders/68/b_2rt18n1w54kfrgsh79rbcw0000gn/T/ipykernel_4570/3414124258.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/mustafazayyat/Downloads/tcp.csv")


In [5]:
df.head()

,PERSON_ID,PERSON_TYPE,CRASH_RECORD_ID,VEHICLE_ID,CRASH_DATE,SEAT_NO,CITY,STATE,ZIPCODE,SEX,...,EMS_RUN_NO,DRIVER_ACTION,DRIVER_VISION,PHYSICAL_CONDITION,PEDPEDAL_ACTION,PEDPEDAL_VISIBILITY,PEDPEDAL_LOCATION,BAC_RESULT,BAC_RESULT VALUE,CELL_PHONE_USE
0,O2289010,DRIVER,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,2183048.0,04/21/2026 11:52:00 PM,NaN,CHICAGO,IL,60637,F,...,NaN,UNKNOWN,UNKNOWN,IMPAIRED - ALCOHOL,NaN,NaN,NaN,TEST REFUSED,NaN,NaN
1,O2289013,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183050.0,04/21/2026 11:43:00 PM,NaN,CHICAGO,IL,60620,M,...,NaN,FAILED TO YIELD,UNKNOWN,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,O2289014,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183052.0,04/21/2026 11:43:00 PM,NaN,CHICAGO,IL,60632,M,...,NaN,NONE,UNKNOWN,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
3,O2289007,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183044.0,04/21/2026 11:40:00 PM,NaN,NaN,NaN,NaN,F,...,NaN,UNKNOWN,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O2289008,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183046.0,04/21/2026 11:40:00 PM,NaN,OAKLAWN,IL,60453,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [6]:
df = pd.read_csv("/Users/mustafazayyat/Downloads/tcp.csv", low_memory=False)

In [7]:
df = df.replace(["UNKNOWN", "NONE", ""], pd.NA)

In [9]:
df.isna().mean().sort_values(ascending=False)

CELL_PHONE_USE           0.999495
BAC_RESULT VALUE         0.998914
EJECTION                 0.995815
EMS_RUN_NO               0.984697
PEDPEDAL_VISIBILITY      0.979905
PEDPEDAL_LOCATION        0.979870
PEDPEDAL_ACTION          0.979869
EMS_AGENCY               0.909279
HOSPITAL                 0.855735
SEAT_NO                  0.798664
DRIVER_ACTION            0.689528
DRIVER_VISION            0.593723
DRIVERS_LICENSE_CLASS    0.515509
PHYSICAL_CONDITION       0.473268
DRIVERS_LICENSE_STATE    0.412804
ZIPCODE                  0.326877
AGE                      0.289828
CITY                     0.275077
STATE                    0.260159
BAC_RESULT               0.202737
VEHICLE_ID               0.020988
AIRBAG_DEPLOYED          0.020207
SEX                      0.017138
SAFETY_EQUIPMENT         0.002764
INJURY_CLASSIFICATION    0.000349
PERSON_TYPE              0.000000
CRASH_DATE               0.000000
CRASH_RECORD_ID          0.000000
PERSON_ID                0.000000
dtype: float64

In [10]:
df = df.drop(columns=[c for c in df.columns if df[c].isna().mean() > 0.8])

In [11]:
df.isna().mean().sort_values(ascending=False)

SEAT_NO                  0.798664
DRIVER_ACTION            0.689528
DRIVER_VISION            0.593723
DRIVERS_LICENSE_CLASS    0.515509
PHYSICAL_CONDITION       0.473268
DRIVERS_LICENSE_STATE    0.412804
ZIPCODE                  0.326877
AGE                      0.289828
CITY                     0.275077
STATE                    0.260159
BAC_RESULT               0.202737
VEHICLE_ID               0.020988
AIRBAG_DEPLOYED          0.020207
SEX                      0.017138
SAFETY_EQUIPMENT         0.002764
INJURY_CLASSIFICATION    0.000349
PERSON_TYPE              0.000000
CRASH_DATE               0.000000
CRASH_RECORD_ID          0.000000
PERSON_ID                0.000000
dtype: float64

In [12]:
df.columns = df.columns.str.lower()

In [13]:
df["crash_record_id"] = df["crash_record_id"].astype(str)

In [14]:
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")
df 

/var/folders/68/b_2rt18n1w54kfrgsh79rbcw0000gn/T/ipykernel_4570/2558081468.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")


,person_id,person_type,crash_record_id,vehicle_id,crash_date,seat_no,city,state,zipcode,sex,age,drivers_license_state,drivers_license_class,safety_equipment,airbag_deployed,injury_classification,driver_action,driver_vision,physical_condition,bac_result
0,O2289010,DRIVER,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,2183048.0,2026-04-21 23:52:00,NaN,CHICAGO,IL,60637,F,38.0,IL,D,USAGE UNKNOWN,"DEPLOYED, FRONT","REPORTED, NOT EVIDENT",<NA>,<NA>,IMPAIRED - ALCOHOL,TEST REFUSED
1,O2289013,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183050.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60620,M,37.0,IL,D,USAGE UNKNOWN,"DEPLOYED, COMBINATION",NO INDICATION OF INJURY,FAILED TO YIELD,<NA>,NORMAL,TEST NOT OFFERED
2,O2289014,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183052.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60632,M,23.0,IL,D,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,NORMAL,TEST NOT OFFERED
3,O2289007,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183044.0,2026-04-21 23:40:00,NaN,NaN,NaN,NaN,F,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,<NA>,TEST NOT OFFERED
4,O2289008,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183046.0,2026-04-21 23:40:00,NaN,OAKLAWN,IL,60453,M,33.0,IL,C,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,NOT OBSCURED,NORMAL,TEST NOT OFFERED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2299858,O1399482,PEDESTRIAN,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,2013-06-01 20:29:00,NaN,CHICAGO,IL,60659,M,3.0,NaN,NaN,NONE PRESENT,NaN,NONINCAPACITATING INJURY,<NA>,<NA>,NORMAL,TEST NOT OFFERED
2299859,O1399483,DRIVER,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,1329848.0,2013-06-01 20:29:00,NaN,CHICAGO,IL,60645,M,19.0,IL,D,SAFETY BELT USED,DID NOT DEPLOY,NO INDICATION OF INJURY,<NA>,NOT OBSCURED,NORMAL,TEST NOT OFFERED
2299860,O595730,DRIVER,a802658be15312809c771559e4f81088cfb226830792a5...,567862.0,2013-03-03 16:48:00,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,FAILED TO YIELD,<NA>,<NA>,TEST NOT OFFERED
2299861,O595731,DRIVER,a802658be15312809c771559e4f81088cfb226830792a5...,567870.0,2013-03-03 16:48:00,NaN,HOFFMAN ESTATES,IL,60169,M,63.0,IL,D,SAFETY BELT USED,DID NOT DEPLOY,"REPORTED, NOT EVIDENT",<NA>,NOT OBSCURED,NORMAL,TEST NOT OFFERED


In [15]:
df["sex"] = df["sex"].str.upper()
df["person_type"] = df["person_type"].str.upper()

In [16]:
df = df.dropna(subset=["crash_date"])
df = df[df["crash_date"] >= "2021-01-01"]
df 

,person_id,person_type,crash_record_id,vehicle_id,crash_date,seat_no,city,state,zipcode,sex,age,drivers_license_state,drivers_license_class,safety_equipment,airbag_deployed,injury_classification,driver_action,driver_vision,physical_condition,bac_result
0,O2289010,DRIVER,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,2183048.0,2026-04-21 23:52:00,NaN,CHICAGO,IL,60637,F,38.0,IL,D,USAGE UNKNOWN,"DEPLOYED, FRONT","REPORTED, NOT EVIDENT",<NA>,<NA>,IMPAIRED - ALCOHOL,TEST REFUSED
1,O2289013,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183050.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60620,M,37.0,IL,D,USAGE UNKNOWN,"DEPLOYED, COMBINATION",NO INDICATION OF INJURY,FAILED TO YIELD,<NA>,NORMAL,TEST NOT OFFERED
2,O2289014,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183052.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60632,M,23.0,IL,D,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,NORMAL,TEST NOT OFFERED
3,O2289007,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183044.0,2026-04-21 23:40:00,NaN,NaN,NaN,NaN,F,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,<NA>,TEST NOT OFFERED
4,O2289008,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183046.0,2026-04-21 23:40:00,NaN,OAKLAWN,IL,60453,M,33.0,IL,C,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,NOT OBSCURED,NORMAL,TEST NOT OFFERED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267836,P230379,PASSENGER,a5b59fe5f189da9d69e493fca558f3f7e9fa7e2e5a17f8...,964731.0,2021-01-01 00:05:00,3.0,NaN,NaN,NaN,M,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,NaN,NaN,NaN,NaN
1267837,O1017884,DRIVER,3b41d999ce80890d8825d8d46ee98570ab1e9c79a0001e...,964732.0,2021-01-01 00:04:00,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,<NA>,TEST NOT OFFERED
1267838,O1019428,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966223.0,2021-01-01 00:00:00,NaN,CHICAGO,IL,60618,M,24.0,IL,D,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,NORMAL,TEST NOT OFFERED
1267839,O1019429,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966232.0,2021-01-01 00:00:00,NaN,CICERO,IL,60804,F,23.0,IL,D,SAFETY BELT USED,DID NOT DEPLOY,NO INDICATION OF INJURY,<NA>,NOT OBSCURED,NORMAL,TEST NOT OFFERED


In [19]:
df2 = pd.read_csv("/Users/mustafazayyat/Downloads/tcc.csv")
df2

/var/folders/68/b_2rt18n1w54kfrgsh79rbcw0000gn/T/ipykernel_4570/373433274.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("/Users/mustafazayyat/Downloads/tcc.csv")


,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,NaN,04/21/2026 11:52:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,1.0,0.0,0.0,23,3,4,41.777841,-87.658580,POINT (-87.658580141377 41.777841455194)
1,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,NaN,04/21/2026 11:43:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0.0,0.0,2.0,0.0,23,3,4,41.736823,-87.585783,POINT (-87.585783420329 41.736822630157)
2,cc736174888117d54b5c4715e258421f458c7e98f606a3...,NaN,04/21/2026 11:40:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0.0,0.0,3.0,0.0,23,3,4,41.771223,-87.741910,POINT (-87.741910387507 41.771222696555)
3,c77a3073aaf27094ffce352ebab93efeca90c662c1c37f...,NaN,04/21/2026 10:10:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,FOUR WAY,...,0.0,0.0,3.0,0.0,22,3,4,41.947068,-87.671357,POINT (-87.671357020322 41.947067504274)
4,3224ccb3df0dfaa9f6ac40eb4bf442d276be190a9b731c...,NaN,04/21/2026 09:35:00 PM,30,UNKNOWN,UNKNOWN,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,21,3,4,41.723127,-87.585289,POINT (-87.585289219624 41.723127203409)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1047833,1d0232afecbdfd01968555aa956a688fd6f55a2bd1984f...,NaN,02/24/2014 07:45:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,DAYLIGHT,TURNING,NOT DIVIDED,...,0.0,0.0,2.0,0.0,19,2,2,41.884016,-87.701143,POINT (-87.701142757538 41.884016475152)
1047834,957783a4787318f005a7dbc920e4c84cb9ac8aa7329a62...,NaN,01/21/2014 07:40:00 AM,30,YIELD,NO CONTROLS,CLEAR,DAYLIGHT,ANGLE,DIVIDED - W/MEDIAN (NOT RAISED),...,1.0,0.0,1.0,0.0,7,3,1,41.760710,-87.561946,POINT (-87.561946030143 41.760710194223)
1047835,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,NaN,01/18/2014 06:14:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,DUSK,PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN BARRIER,...,0.0,0.0,2.0,0.0,18,7,1,41.885610,-87.638756,POINT (-87.638756189808 41.885609917063)
1047836,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,06/01/2013 08:29:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PEDESTRIAN,NOT DIVIDED,...,1.0,0.0,1.0,0.0,20,7,6,42.005145,-87.689969,POINT (-87.689968732521 42.005144534455)


In [21]:
df2 = df2.replace(["UNKNOWN", "NONE", ""], pd.NA)

In [22]:
df.isna().mean().sort_values(ascending=False)

seat_no                  0.801738
driver_action            0.694783
driver_vision            0.635901
drivers_license_class    0.541532
physical_condition       0.491928
drivers_license_state    0.417944
zipcode                  0.329787
age                      0.294695
city                     0.284674
state                    0.267579
bac_result               0.200098
vehicle_id               0.021810
airbag_deployed          0.021082
sex                      0.019017
safety_equipment         0.002588
injury_classification    0.000196
person_type              0.000000
crash_date               0.000000
crash_record_id          0.000000
person_id                0.000000
dtype: float64

In [23]:
df2.columns = df2.columns.str.lower()

In [24]:
df2["crash_record_id"] = df2["crash_record_id"].astype(str)
df2["crash_date"] = pd.to_datetime(df2["crash_date"], errors="coerce")

/var/folders/68/b_2rt18n1w54kfrgsh79rbcw0000gn/T/ipykernel_4570/2505628314.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df2["crash_date"] = pd.to_datetime(df2["crash_date"], errors="coerce")


In [25]:
df

,person_id,person_type,crash_record_id,vehicle_id,crash_date,seat_no,city,state,zipcode,sex,age,drivers_license_state,drivers_license_class,safety_equipment,airbag_deployed,injury_classification,driver_action,driver_vision,physical_condition,bac_result
0,O2289010,DRIVER,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,2183048.0,2026-04-21 23:52:00,NaN,CHICAGO,IL,60637,F,38.0,IL,D,USAGE UNKNOWN,"DEPLOYED, FRONT","REPORTED, NOT EVIDENT",<NA>,<NA>,IMPAIRED - ALCOHOL,TEST REFUSED
1,O2289013,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183050.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60620,M,37.0,IL,D,USAGE UNKNOWN,"DEPLOYED, COMBINATION",NO INDICATION OF INJURY,FAILED TO YIELD,<NA>,NORMAL,TEST NOT OFFERED
2,O2289014,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183052.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60632,M,23.0,IL,D,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,NORMAL,TEST NOT OFFERED
3,O2289007,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183044.0,2026-04-21 23:40:00,NaN,NaN,NaN,NaN,F,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,<NA>,TEST NOT OFFERED
4,O2289008,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183046.0,2026-04-21 23:40:00,NaN,OAKLAWN,IL,60453,M,33.0,IL,C,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,NOT OBSCURED,NORMAL,TEST NOT OFFERED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267836,P230379,PASSENGER,a5b59fe5f189da9d69e493fca558f3f7e9fa7e2e5a17f8...,964731.0,2021-01-01 00:05:00,3.0,NaN,NaN,NaN,M,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,NaN,NaN,NaN,NaN
1267837,O1017884,DRIVER,3b41d999ce80890d8825d8d46ee98570ab1e9c79a0001e...,964732.0,2021-01-01 00:04:00,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,<NA>,TEST NOT OFFERED
1267838,O1019428,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966223.0,2021-01-01 00:00:00,NaN,CHICAGO,IL,60618,M,24.0,IL,D,USAGE UNKNOWN,DEPLOYMENT UNKNOWN,NO INDICATION OF INJURY,<NA>,<NA>,NORMAL,TEST NOT OFFERED
1267839,O1019429,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966232.0,2021-01-01 00:00:00,NaN,CICERO,IL,60804,F,23.0,IL,D,SAFETY BELT USED,DID NOT DEPLOY,NO INDICATION OF INJURY,<NA>,NOT OBSCURED,NORMAL,TEST NOT OFFERED


In [26]:
df2 = df2.dropna(subset=["crash_date"])
df2 = df2[df2["crash_date"] >= "2021-01-01"]
df2 

,crash_record_id,crash_date_est_i,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,crash_month,latitude,longitude,location
0,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,NaN,2026-04-21 23:52:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,1.0,0.0,0.0,23,3,4,41.777841,-87.658580,POINT (-87.658580141377 41.777841455194)
1,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,NaN,2026-04-21 23:43:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0.0,0.0,2.0,0.0,23,3,4,41.736823,-87.585783,POINT (-87.585783420329 41.736822630157)
2,cc736174888117d54b5c4715e258421f458c7e98f606a3...,NaN,2026-04-21 23:40:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0.0,0.0,3.0,0.0,23,3,4,41.771223,-87.741910,POINT (-87.741910387507 41.771222696555)
3,c77a3073aaf27094ffce352ebab93efeca90c662c1c37f...,NaN,2026-04-21 22:10:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,FOUR WAY,...,0.0,0.0,3.0,0.0,22,3,4,41.947068,-87.671357,POINT (-87.671357020322 41.947067504274)
4,3224ccb3df0dfaa9f6ac40eb4bf442d276be190a9b731c...,NaN,2026-04-21 21:35:00,30,<NA>,<NA>,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,21,3,4,41.723127,-87.585289,POINT (-87.585289219624 41.723127203409)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581096,90e627d390edd0adcfb35bfcc5c12c7eeafa68d04b66d0...,NaN,2021-01-01 00:10:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,0,6,1,41.812153,-87.684680,POINT (-87.68468042746 41.812153033185)
581097,4ef013035e1af1eb9ea11f83a1251e7ee80b5e35e916d9...,NaN,2021-01-01 00:05:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,1.0,0.0,1.0,0.0,0,6,1,42.007271,-87.673711,POINT (-87.673710862369 42.007270860381)
581098,a5b59fe5f189da9d69e493fca558f3f7e9fa7e2e5a17f8...,NaN,2021-01-01 00:05:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,0,6,1,41.903129,-87.686909,POINT (-87.686909128207 41.903128739116)
581099,3b41d999ce80890d8825d8d46ee98570ab1e9c79a0001e...,NaN,2021-01-01 00:04:00,10,NO CONTROLS,NO CONTROLS,CLOUDY/OVERCAST,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,OTHER,...,0.0,0.0,1.0,0.0,0,6,1,41.937369,-87.707427,POINT (-87.707427053603 41.937368594943)


In [27]:
dfm = df.merge(df2, on="crash_record_id", how="left")
dfm

,person_id,person_type,crash_record_id,vehicle_id,crash_date_x,seat_no,city,state,zipcode,sex,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,crash_month,latitude,longitude,location
0,O2289010,DRIVER,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,2183048.0,2026-04-21 23:52:00,NaN,CHICAGO,IL,60637,F,...,0.0,1.0,0.0,0.0,23,3,4,41.777841,-87.658580,POINT (-87.658580141377 41.777841455194)
1,O2289013,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183050.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60620,M,...,0.0,0.0,2.0,0.0,23,3,4,41.736823,-87.585783,POINT (-87.585783420329 41.736822630157)
2,O2289014,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183052.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60632,M,...,0.0,0.0,2.0,0.0,23,3,4,41.736823,-87.585783,POINT (-87.585783420329 41.736822630157)
3,O2289007,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183044.0,2026-04-21 23:40:00,NaN,NaN,NaN,NaN,F,...,0.0,0.0,3.0,0.0,23,3,4,41.771223,-87.741910,POINT (-87.741910387507 41.771222696555)
4,O2289008,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183046.0,2026-04-21 23:40:00,NaN,OAKLAWN,IL,60453,M,...,0.0,0.0,3.0,0.0,23,3,4,41.771223,-87.741910,POINT (-87.741910387507 41.771222696555)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267836,P230379,PASSENGER,a5b59fe5f189da9d69e493fca558f3f7e9fa7e2e5a17f8...,964731.0,2021-01-01 00:05:00,3.0,NaN,NaN,NaN,M,...,0.0,0.0,2.0,0.0,0,6,1,41.903129,-87.686909,POINT (-87.686909128207 41.903128739116)
1267837,O1017884,DRIVER,3b41d999ce80890d8825d8d46ee98570ab1e9c79a0001e...,964732.0,2021-01-01 00:04:00,NaN,NaN,NaN,NaN,X,...,0.0,0.0,1.0,0.0,0,6,1,41.937369,-87.707427,POINT (-87.707427053603 41.937368594943)
1267838,O1019428,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966223.0,2021-01-01 00:00:00,NaN,CHICAGO,IL,60618,M,...,0.0,0.0,3.0,0.0,0,6,1,41.944745,-87.706314,POINT (-87.706314298452 41.944745276013)
1267839,O1019429,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966232.0,2021-01-01 00:00:00,NaN,CICERO,IL,60804,F,...,0.0,0.0,3.0,0.0,0,6,1,41.944745,-87.706314,POINT (-87.706314298452 41.944745276013)


In [28]:
dfm = dfm.drop_duplicates()
dfm

,person_id,person_type,crash_record_id,vehicle_id,crash_date_x,seat_no,city,state,zipcode,sex,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,crash_month,latitude,longitude,location
0,O2289010,DRIVER,6f905467f8ad92a9f361d854b39b1540b2db3386ee7ef2...,2183048.0,2026-04-21 23:52:00,NaN,CHICAGO,IL,60637,F,...,0.0,1.0,0.0,0.0,23,3,4,41.777841,-87.658580,POINT (-87.658580141377 41.777841455194)
1,O2289013,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183050.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60620,M,...,0.0,0.0,2.0,0.0,23,3,4,41.736823,-87.585783,POINT (-87.585783420329 41.736822630157)
2,O2289014,DRIVER,1a8862d1b7194dbcc4c85e2452b48647ec0125d6d7ac7e...,2183052.0,2026-04-21 23:43:00,NaN,CHICAGO,IL,60632,M,...,0.0,0.0,2.0,0.0,23,3,4,41.736823,-87.585783,POINT (-87.585783420329 41.736822630157)
3,O2289007,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183044.0,2026-04-21 23:40:00,NaN,NaN,NaN,NaN,F,...,0.0,0.0,3.0,0.0,23,3,4,41.771223,-87.741910,POINT (-87.741910387507 41.771222696555)
4,O2289008,DRIVER,cc736174888117d54b5c4715e258421f458c7e98f606a3...,2183046.0,2026-04-21 23:40:00,NaN,OAKLAWN,IL,60453,M,...,0.0,0.0,3.0,0.0,23,3,4,41.771223,-87.741910,POINT (-87.741910387507 41.771222696555)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267836,P230379,PASSENGER,a5b59fe5f189da9d69e493fca558f3f7e9fa7e2e5a17f8...,964731.0,2021-01-01 00:05:00,3.0,NaN,NaN,NaN,M,...,0.0,0.0,2.0,0.0,0,6,1,41.903129,-87.686909,POINT (-87.686909128207 41.903128739116)
1267837,O1017884,DRIVER,3b41d999ce80890d8825d8d46ee98570ab1e9c79a0001e...,964732.0,2021-01-01 00:04:00,NaN,NaN,NaN,NaN,X,...,0.0,0.0,1.0,0.0,0,6,1,41.937369,-87.707427,POINT (-87.707427053603 41.937368594943)
1267838,O1019428,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966223.0,2021-01-01 00:00:00,NaN,CHICAGO,IL,60618,M,...,0.0,0.0,3.0,0.0,0,6,1,41.944745,-87.706314,POINT (-87.706314298452 41.944745276013)
1267839,O1019429,DRIVER,1448e416d21b7c264b758414a1c2aa86995e8a297f3159...,966232.0,2021-01-01 00:00:00,NaN,CICERO,IL,60804,F,...,0.0,0.0,3.0,0.0,0,6,1,41.944745,-87.706314,POINT (-87.706314298452 41.944745276013)


In [58]:
dfm["crash_record_id"].isna().sum()

np.int64(0)

In [59]:
# Any injury
dfm["any_injury"] = (
    (dfm["injuries_non_incapacitating"] > 0) |
    (dfm["injuries_incapacitating"] > 0) |
    (dfm["injuries_fatal"] > 0)
).astype(int)

# Severe only
dfm["severe_injury"] = (
    (dfm["injuries_incapacitating"] > 0) |
    (dfm["injuries_fatal"] > 0)
).astype(int)

In [60]:
df2 = dfm.groupby("crash_record_id").agg({
    "any_injury": "max",
    "person_id": "count",
    "posted_speed_limit": "first",
    "weather_condition": "first",
    "lighting_condition": "first",
    "traffic_control_device": "first",
    "roadway_surface_cond": "first",
    "crash_hour": "first",
    "crash_day_of_week": "first",
    "crash_month": "first",
    "latitude": "first",
    "longitude": "first"
}).rename(columns={
    "person_id": "num_people"
}).reset_index()

In [61]:
df2["weather_group"] = df2["weather_condition"].replace({
    "CLEAR": "CLEAR",
    "CLOUDY/OVERCAST": "CLEAR",
    "RAIN": "BAD",
    "SNOW": "BAD",
    "FOG/SMOKE/HAZE": "BAD",
    "SLEET/HAIL": "BAD",
    "FREEZING RAIN/DRIZZLE": "BAD",
    "BLOWING SNOW": "BAD",
    "BLOWING SAND, SOIL, DIRT": "BAD"
})

In [62]:
df2["lighting_group"] = df2["lighting_condition"].replace({
    "DAYLIGHT": "DAY",
    "DARKNESS": "NIGHT",
    "DARKNESS, LIGHTED ROAD": "NIGHT",
    "DUSK": "LOW_LIGHT",
    "DAWN": "LOW_LIGHT"
})

In [63]:
df2["speed_bin"] = pd.cut(
    df2["posted_speed_limit"],
    bins=[0, 25, 35, 45, 100],
    labels=["LOW", "MEDIUM", "HIGH", "VERY_HIGH"]
)

In [64]:
df2["is_night"] = df2["crash_hour"].isin([20,21,22,23,0,1,2,3,4,5])

In [65]:
df2.groupby("weather_group")["any_injury"].mean()


weather_group
BAD                       0.114572
CLEAR                     0.109512
OTHER                     0.112281
SEVERE CROSS WIND GATE    0.114286
Name: any_injury, dtype: float64

In [66]:
df2.groupby("lighting_group")["any_injury"].mean()

lighting_group
DAY          0.098317
LOW_LIGHT    0.120097
NIGHT        0.133905
Name: any_injury, dtype: float64

In [67]:
df2.groupby("speed_bin")["any_injury"].mean()

/var/folders/68/b_2rt18n1w54kfrgsh79rbcw0000gn/T/ipykernel_4570/1334704479.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2.groupby("speed_bin")["any_injury"].mean()


speed_bin
LOW          0.065816
MEDIUM       0.112152
HIGH         0.147398
VERY_HIGH    0.139535
Name: any_injury, dtype: float64

In [68]:
from sklearn.linear_model import LogisticRegression

X = pd.get_dummies(df2[[
    "posted_speed_limit",
    "weather_group",
    "lighting_group",
    "is_night"
]], drop_first=True)

y = df2["any_injury"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

LogisticRegression(max_iter=1000)

In [70]:
injutry_prob = model.predict_proba(X)[:, 1]
injutry_prob

array([0.1006204 , 0.1006204 , 0.06700482, ..., 0.13214767, 0.15044106,
       0.09757634])

In [73]:
predict = model.predict(X)
predict

array([0, 0, 0, ..., 0, 0, 0])

In [74]:
from sklearn.metrics import accuracy_score, roc_auc_score

accuracy = accuracy_score(y, predict)
auc = roc_auc_score(y, injutry_prob)

accuracy, auc

(0.8956930140622548, np.float64(0.5867387899345122))

In [77]:
dfm.to_csv("/Users/mustafazayyat/Downloads/merged_dataset.csv", index=False)
df2.to_csv("/Users/mustafazayyat/Downloads/crashes_dataset.csv", index=False)
df.to_csv("/Users/mustafazayyat/Downloads/people_dataset.csv", index=False)